# bsc_01 — M0 Headroom  ⭐ **GATE 1 (cổng chặn chính)**

**Đây là thứ quan trọng nhất của Stage 1.** Không có trong plan doc. Nó hỏi đúng câu
ROI cascade đã quên: **phần thưởng lớn cỡ nào — TRƯỚC khi xây bất cứ thứ gì.**

Chạy trên **OAI-ZIB test (103 ca)** — nguồn duy nhất có GT xương. ~1h CPU, **không GPU**.

Đo, với mỗi ca × mỗi lớp sụn:
1. Trường độ dày GT (từ tia dọc pháp tuyến xương).
2. Phân rã **error mass** của ResEnc (B0) theo bin độ dày — hai chiều.
3. Counterfactual: nếu triệt tiêu lỗi ở bin {absent, ≤0.5, ≤1.0}, ASSD cải thiện bao nhiêu.
4. M0c: bậc thang z của GT.

**GATE 1:**
- ✅ PROCEED: `ΔASSD_prize ≥ 0.08mm` **và** thin+absent ≥ 35% error mass
- ⚠️ RESCOPE: `ΔASSD_prize ∈ [0.04, 0.08)` → viết lại mục tiêu quanh presence/thickness
- ❌ STOP: `ΔASSD_prize < 0.04mm` **hoặc** bậc thang z chiếm ưu thế

**Xác suất thật thà: 50/50.** Sụn đùi phần lớn dày 2–3mm. Nhưng error mass (không phải
diện tích) mới đáng kể — vùng mỏng khó bất tương xứng.


### Cell config (giống bsc_00)

In [ ]:
# ============================================================
# CELL CONFIG CHUAN - tai dung o MOI notebook bsc_*
# Drive-first: MOI artifact nam duoi BSC_ROOT. KHONG ghi vao /content/.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/<user>/nnUnet-OAI"   # <-- doi thanh repo cua ban
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
sys.path.insert(0, REPO_DIR)

# Thu can cho Colab (may local da co scipy/skimage/numpy)
!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"          # goc artifact - TAT CA nam duoi day
os.makedirs(BSC_ROOT, exist_ok=True)
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

# Duong du lieu cu (READ-ONLY - khong bao gio ghi de)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"   # <-- kiem lai duong nay
print("BSC_ROOT =", BSC_ROOT)
print("Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.")

## Chạy M0 trên 103 ca test

Cần **prediction B0 trên test set** (không phải CV). Nếu chưa có, chạy inference d020
fold_0 trên `imagesTs` (103 ca) rồi lưu vào `BSC_ROOT/baselines/B0_test_pred/`.
Đây là lần **duy nhất** dùng test set trước Gate 4 — chỉ để đo headroom, không tune gì.

In [ ]:
# ---- Sinh B0_test_pred neu chua co: chay B0 (250ep fold_0) tren 103 anh test ----
# Day la buoc THIEU trong notebook goc: markdown cell tren mo ta nhung khong co code.
# B0_test_pred KHONG nam trong zip (zip chi co CV validation pred), phai TU chay inference.
# ⭐ BUOC NAY DUNG GPU - bat runtime GPU cho nhanh (khac vong metric CPU o Gate 0/M0).
import os, glob
B0_PRED = f"{BSC_ROOT}/baselines/B0_test_pred"
os.makedirs(B0_PRED, exist_ok=True)

n_img  = len(glob.glob(f"{RAW}/imagesTs/*_0000.nii.gz"))
n_have = len(glob.glob(f"{B0_PRED}/oaizib_*.nii.gz"))
print(f"Anh test: {n_img} | prediction da co: {n_have}")
assert n_img > 0, f"Khong thay anh test o {RAW}/imagesTs (can *_0000.nii.gz)"

if n_have < n_img:
    # nnUNet doc model tu $nnUNet_results. Zip da bung full folder trainer vao ds020.
    os.environ["nnUNet_results"]      = f"{BSC_ROOT}/baselines/ds020"
    os.environ["nnUNet_raw"]          = os.path.dirname(RAW)        # nnUNet doi co, khong dung khi predict
    os.environ["nnUNet_preprocessed"] = "/content/nnunet_prep_tmp"  # tmp, khong dung khi predict
    os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)

    # Model folder phai du dataset.json + plans.json + checkpoint truoc khi chay
    mdir = glob.glob(f"{os.environ['nnUNet_results']}/Dataset020_KneeUnion/"
                     f"nnUNetTrainer_250epochs*ResEnc*3d_fullres")[0]
    for need in ["dataset.json", "plans.json", "fold_0/checkpoint_best.pth"]:
        assert os.path.exists(f"{mdir}/{need}"), f"Thieu {need} trong {mdir}"
    print("Model OK:", os.path.basename(mdir))

    import importlib.util
    if importlib.util.find_spec("nnunetv2") is None:
        !pip install -q nnunetv2

    import torch
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (rat cham - bat GPU!)"
    print("Device:", dev)

    # -tr/-p/-c lay tu ten folder: nnUNetTrainer_250epochs__nnUNetResEncUNetLPlans__3d_fullres
    !nnUNetv2_predict -i "{RAW}/imagesTs" -o "{B0_PRED}" -d 020 -c 3d_fullres -tr nnUNetTrainer_250epochs -p nnUNetResEncUNetLPlans -f 0 -chk checkpoint_best.pth

    n_have = len(glob.glob(f"{B0_PRED}/oaizib_*.nii.gz"))

assert n_have >= n_img, f"Inference chua du: {n_have}/{n_img}. Xem log nnUNet ben tren."
print(f"B0_test_pred san sang: {n_have} prediction o {B0_PRED}")

In [ ]:
import glob, json, os
import numpy as np
from tqdm import tqdm
from bsc import io_utils, headroom, core
from bsc.core import RayConfig

RAW_TS_IMG = f"{RAW}/imagesTs"          # 103 anh test
RAW_TS_LAB = f"{RAW}/labelsTs"          # 103 GT test (co xuong+sun, [0..5])
B0_PRED    = f"{BSC_ROOT}/baselines/B0_test_pred"
CKPT       = f"{BSC_ROOT}/runs/M0_percase.jsonl"   # Drive-first: resume khi dut ket noi

CART = {"femoral_cart": 2, "med_tib_cart": 4}   # sun dui + chay trong (§3.1)
BONE = {"femoral_cart": 1, "med_tib_cart": 3}   # xuong tuong ung lam neo he toa do

cfg = RayConfig()
cases = io_utils.list_cases(RAW_TS_IMG)
print(f"{len(cases)} ca test")

# --- Resume: doc ket qua per-case da co ---------------------------------------
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
results = {c: {"prize": [], "mass": [], "stair": []} for c in CART}
done = set()
if os.path.exists(CKPT):
    with open(CKPT) as f:
        for line in f:
            r = json.loads(line)
            done.add((r["case"], r["cls"]))
            results[r["cls"]]["prize"].append(r["prize"])
            results[r["cls"]]["mass"].append(r["mass"])
            results[r["cls"]]["stair"].append(r["stair"])
    print(f"Tiep tuc: da co {len(done)} (ca,lop) trong {CKPT}")

def _clean(o):   # numpy -> python cho json
    if isinstance(o, dict):  return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(x) for x in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o

n_skip = 0
with open(CKPT, "a") as fh:
    for cid in tqdm(cases, desc="M0"):
        prf = f"{B0_PRED}/{cid}.nii.gz"
        if not os.path.exists(prf):
            n_skip += 1
            continue
        need = [c for c in CART if (cid, c) not in done]
        if not need:
            continue
        gt, sp = io_utils.load_nii(f"{RAW_TS_LAB}/{cid}.nii.gz")
        pr, _ = io_utils.load_nii(prf)
        for c in need:
            bone = (gt == BONE[c])
            gt_c, pr_c = (gt == CART[c]), (pr == CART[c])
            if not gt_c.any() or not bone.any():
                continue
            # Thickness field tinh MOT lan, dung cho ca prize lan error_mass (bo marching-cubes 2x)
            tf = headroom.gt_thickness_per_node(bone, gt_c, sp, cfg)
            row = {
                "case": cid, "cls": c,
                "prize": _clean(headroom.prize_counterfactual(gt_c, pr_c, bone, sp, cfg, thickness=tf)),
                "mass":  _clean(headroom.error_mass_by_thickness(gt_c, pr_c, bone, sp, cfg, thickness=tf)),
                "stair": _clean(headroom.gt_staircase_z_vs_inplane(gt_c, sp)),
            }
            results[c]["prize"].append(row["prize"])
            results[c]["mass"].append(row["mass"])
            results[c]["stair"].append(row["stair"])
            fh.write(json.dumps(row) + "\n")
            fh.flush()   # ghi ngay: dut ket noi van con du lieu

if n_skip:
    print(f"CANH BAO: {n_skip} ca thieu prediction o {B0_PRED} (chay lai cell inference?)")
assert any(results[c]["prize"] for c in CART), (
    f"Khong tinh duoc ca nao. Thieu pred: {n_skip}. Kiem B0_test_pred va labelsTs."
)
print(f"Xong M0. Per-case da ghi: {CKPT}")

## GATE 1 — quyết định

In [ ]:
import numpy as np

for c in CART:
    g = headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])
    print(f"\n=== {c} ===")
    print(f"  ΔASSD_prize = {g['d_assd_prize_mean_mm']:.4f} mm  CI {g['d_assd_prize_ci']}")
    print(f"  thin+absent error mass = {g['thin_absent_error_mass_frac_mean']:.1%}")
    print(f"  QUYET DINH (luat hien hanh): {g['decision']}")

    # Bang error mass theo bin
    print(f"  {'bin':<10}{'N':>8}{'mean_d':>9}{'mass%':>8}")
    n_by_bin = {}
    for nm in core.THICKNESS_NAMES:
        fr = np.mean([m['per_bin'][nm]['frac_error_mass'] for m in results[c]['mass']])
        nn = np.mean([m['per_bin'][nm]['n'] for m in results[c]['mass']])
        md_ = np.mean([m['per_bin'][nm]['mean_dist_mm'] for m in results[c]['mass']])
        n_by_bin[nm] = nn
        print(f"  {nm:<10}{nn:>8.0f}{md_:>9.3f}{fr:>7.1%}")

    # --- CHAN DOAN BO SUNG: ty so TAP TRUNG (khong phai luat quyet dinh) -------
    # Docstring headroom noi ly do la "vung mong gom loi BAT TUONG XUNG so voi dien tich",
    # nhung nguong 35% lai do TUYET DOI. Hai thu khac nhau => in ca hai, quyet dinh sau.
    thin = ("absent", "<=0.5mm", "<=1.0mm")
    n_thin = sum(n_by_bin[nm] for nm in thin)
    n_all = sum(n_by_bin.values())
    area_frac = n_thin / n_all if n_all else np.nan
    mass_frac = g['thin_absent_error_mass_frac_mean']
    ratio = mass_frac / area_frac if area_frac else np.nan
    print(f"  --- chan doan (khong phai luat) ---")
    print(f"  thin+absent chiem {area_frac:.1%} DIEN TICH nhung {mass_frac:.1%} ERROR MASS")
    print(f"  => ty so TAP TRUNG = {ratio:.2f}x   (>1 = loi don vao vung mong bat tuong xung)")

    stair = np.nanmean([s['z_over_inplane_std'] for s in results[c]['stair']])
    print(f"  M0c bac thang z/in-plane = {stair:.3f}  (>>1 => nhieu z chiem uu the)")

## GATE 1 phu — `ε_metric`: prize co phu thuoc DINH NGHIA metric khong?

Review §5.3 / §6.5. Prize la hieu TRONG cung mot impl nen offset hang so **triet tieu** —
NHUNG counterfactual chi bo voxel o vung MONG/RIA, noi offset co the khac phan than. Phai do:

`ε_metric = Prize_scipy − Prize_sitk  ( = δ_B − δ_M )`

**Vi sao quan trong voi femoral:** δ femoral = **+0.064mm** (hang so, do o bsc_00 cell 3b),
**cung co voi chinh prize femoral 0.058mm**. Neu offset khong cancel sach thi prize femoral
xe dich dang ke. Nguoc lai δ med_tib ≈ +0.004mm => so med_tib gan nhu chac chan vung.

Doc ket qua:
- `|ε|` **nho** so voi prize + hai prize cung dau => ket luan **vung**, tin duoc.
- `|ε|` **cung co** voi prize => ket luan phu thuoc dinh nghia metric => phai chot MOT
  contract va tinh lai TAT CA baseline theo no (§9.1-2 review) truoc khi quyet dinh.

In [ ]:
import json, os
import numpy as np
import SimpleITK as sitk
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm
from bsc import io_utils, headroom, core, metrics
from bsc.core import RayConfig

# --- Cell TU DUNG DOC LAP: chi can BSC_ROOT + RAW tu cell config -------------
RAW_TS_IMG = f"{RAW}/imagesTs"
RAW_TS_LAB = f"{RAW}/labelsTs"
B0_PRED    = f"{BSC_ROOT}/baselines/B0_test_pred"
EPS_CKPT   = f"{BSC_ROOT}/runs/M0_eps_metric.jsonl"

CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}
cfg = RayConfig()
cases = io_utils.list_cases(RAW_TS_IMG)

# None = TAT CA (khuyen nghi: review §4.1 doi full set cho so cuoi cung).
# Resume theo (case,cls) nen lan chay truoc (15 ca) duoc tai su dung, chi chay phan thieu.
N_EPS = None
PRIZE_BINS = ("absent", "<=0.5mm", "<=1.0mm")

def _prize_from(dist_gt, dist_pr, thick_gt, thick_pr):
    """Cong thuc prize Y HET headroom.prize_counterfactual, tach ra de dung cho ca 2 impl."""
    bins_gt = core.assign_thickness_bin(thick_gt)
    bins_pr = core.assign_thickness_bin(thick_pr)
    idx = [core.THICKNESS_NAMES.index(b) for b in PRIZE_BINS]
    keep_gt, keep_pr = ~np.isin(bins_gt, idx), ~np.isin(bins_pr, idx)
    base = np.concatenate([dist_gt, dist_pr])
    kept = np.concatenate([dist_gt[keep_gt], dist_pr[keep_pr]])
    base, kept = base[np.isfinite(base)], kept[np.isfinite(kept)]
    if base.size == 0:
        return np.nan
    return float(base.mean()) - float(kept.sum() / base.size)   # assd_base - assd_prize

def _prize_scipy(gt_c, pr_c, verts, thick_node, sp):
    sg, sp_ = metrics.surface_mask(gt_c), metrics.surface_mask(pr_c)
    if not sg.any() or not sp_.any():
        return np.nan
    d_gt = distance_transform_edt(~sp_, sampling=sp)[sg]     # GT surf -> PRED surf
    d_pr = distance_transform_edt(~sg, sampling=sp)[sp_]     # PRED surf -> GT surf
    t_gt = core.nearest_surface_value(verts, thick_node, core.voxel_centers_mm(sg, sp))
    t_pr = core.nearest_surface_value(verts, thick_node, core.voxel_centers_mm(sp_, sp))
    return _prize_from(d_gt, d_pr, t_gt, t_pr)

def _prize_sitk(gt_c, pr_c, verts, thick_node, sp):
    """Cung counterfactual nhung dung LabelContour + SignedMaurer (dinh nghia cu)."""
    sp_xyz = (sp[2], sp[1], sp[0])
    gi = sitk.GetImageFromArray(gt_c.astype(np.uint8)); gi.SetSpacing(sp_xyz)
    pi = sitk.GetImageFromArray(pr_c.astype(np.uint8)); pi.SetSpacing(sp_xyz)
    gs = sitk.GetArrayFromImage(sitk.LabelContour(gi)).astype(bool)
    ps = sitk.GetArrayFromImage(sitk.LabelContour(pi)).astype(bool)
    if not gs.any() or not ps.any():
        return np.nan
    gdm = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(
        gi, squaredDistance=False, useImageSpacing=True)))
    pdm = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(
        pi, squaredDistance=False, useImageSpacing=True)))
    d_gt, d_pr = pdm[gs], gdm[ps]                            # GT surf -> PRED ; PRED -> GT
    t_gt = core.nearest_surface_value(verts, thick_node, core.voxel_centers_mm(gs, sp))
    t_pr = core.nearest_surface_value(verts, thick_node, core.voxel_centers_mm(ps, sp))
    return _prize_from(d_gt, d_pr, t_gt, t_pr)

# --- Resume ------------------------------------------------------------------
os.makedirs(os.path.dirname(EPS_CKPT), exist_ok=True)
rows, done = [], set()
if os.path.exists(EPS_CKPT):
    with open(EPS_CKPT) as f:
        for line in f:
            r = json.loads(line); rows.append(r); done.add((r["case"], r["cls"]))
    print(f"Tiep tuc: da co {len(done)} (ca,lop)")

sel = [c for c in cases if os.path.exists(f"{B0_PRED}/{c}.nii.gz")]
if N_EPS:
    sel = sel[:N_EPS]
assert sel, f"Khong thay prediction nao o {B0_PRED} - chay cell inference truoc."
todo = [c for c in sel if any((c, cl) not in done for cl in CART)]
print(f"eps_metric tren {len(sel)} ca ({len(todo)} ca con phai chay, ~45s/ca)")

with open(EPS_CKPT, "a") as fh:
    for cid in tqdm(todo, desc="eps_metric"):
        need = [c for c in CART if (cid, c) not in done]
        if not need:
            continue
        gt, sp = io_utils.load_nii(f"{RAW_TS_LAB}/{cid}.nii.gz")
        pr, _ = io_utils.load_nii(f"{B0_PRED}/{cid}.nii.gz")
        for c in need:
            bone, gt_c, pr_c = (gt == BONE[c]), (gt == CART[c]), (pr == CART[c])
            if not gt_c.any() or not bone.any() or not pr_c.any():
                continue
            verts, _, thick_node = headroom.gt_thickness_per_node(bone, gt_c, sp, cfg)
            r = {"case": cid, "cls": c,
                 "prize_scipy": _prize_scipy(gt_c, pr_c, verts, thick_node, sp),
                 "prize_sitk":  _prize_sitk(gt_c, pr_c, verts, thick_node, sp)}
            rows.append(r)
            fh.write(json.dumps(r) + "\n"); fh.flush()

# --- Bao cao ε ---------------------------------------------------------------
hdr = (f"{'lop':<14}{'n':>4}{'Prize_scipy':>13}{'Prize_sitk':>12}"
       f"{'eps':>11}{'|eps|/prize':>13}{'boot95%CI(eps)':>22}")
print("\n" + hdr)
for c in CART:
    ps_ = np.array([r["prize_scipy"] for r in rows if r["cls"] == c], float)
    pk_ = np.array([r["prize_sitk"] for r in rows if r["cls"] == c], float)
    ok = np.isfinite(ps_) & np.isfinite(pk_)
    ps_, pk_ = ps_[ok], pk_[ok]
    if ps_.size == 0:
        print(f"{c:<14}  (khong co ca hop le)")
        continue
    eps = ps_ - pk_
    b = metrics.paired_bootstrap(np.zeros_like(eps), eps, n_boot=10000, seed=0)
    ratio = abs(eps.mean()) / abs(ps_.mean()) if ps_.mean() else np.nan
    ci = f"[{b['ci_low']:+.4f},{b['ci_high']:+.4f}]"
    print(f"{c:<14}{ps_.size:>4}{ps_.mean():>13.4f}{pk_.mean():>12.4f}"
          f"{eps.mean():>+11.4f}{ratio:>12.0%}{ci:>22}")

print("\nDoc ket qua:")
print("  |eps|/prize < ~20% va hai prize cung dau => VUNG. Ket luan Gate 1 khong phu thuoc")
print("     dinh nghia metric. Tin so scipy, di tiep.")
print("  |eps|/prize lon (>~50%) hoac doi dau     => prize PHU THUOC dinh nghia. Phai chot")
print("     mot contract metric (§9.1-2 review) va tinh lai baseline truoc khi quyet dinh.")

In [ ]:
# ---- DOI SOAT hai bang: Gate 1 (103 ca) vs eps_metric (15 ca) ----------------
# Cau hoi: chenh lech 0.0582 vs 0.0550 la do TAP CA hay do PIPELINE khac nhau?
# Ca hai checkpoint deu luu gia tri PER-CASE => join theo (case, cls) la biet CHAC,
# khong can tinh lai gi. Neu tren cung mot ca ma hai ben ra so giong het => pipeline
# giong nhau, chenh lech thuan tuy do tap ca.
import json, os
import numpy as np

M0_CKPT  = f"{BSC_ROOT}/runs/M0_percase.jsonl"
EPS_CKPT = f"{BSC_ROOT}/runs/M0_eps_metric.jsonl"

m0 = {}
with open(M0_CKPT) as f:
    for line in f:
        r = json.loads(line)
        m0[(r["case"], r["cls"])] = r["prize"]["d_assd_prize_mm"]

ep = {}
with open(EPS_CKPT) as f:
    for line in f:
        r = json.loads(line)
        ep[(r["case"], r["cls"])] = r["prize_scipy"]

shared = sorted(set(m0) & set(ep))
print(f"Ca chung giua hai bang: {len(shared)}\n")

print(f"{'lop':<14}{'n_chung':>8}{'A:Gate1':>10}{'B:eps':>10}{'max|A-B|':>11}  ket luan")
for c in ("femoral_cart", "med_tib_cart"):
    k = [x for x in shared if x[1] == c]
    a = np.array([m0[x] for x in k], float)
    b = np.array([ep[x] for x in k], float)
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if a.size == 0:
        print(f"{c:<14}  (khong co ca chung)")
        continue
    dmax = np.max(np.abs(a - b))
    verdict = "PIPELINE GIONG NHAU" if dmax < 1e-6 else "!! PIPELINE KHAC - dieu tra"
    print(f"{c:<14}{a.size:>8}{a.mean():>10.4f}{b.mean():>10.4f}{dmax:>11.2e}  {verdict}")

# Cung MOT pipeline (cot A), so tap 15 ca vs toan bo => luong hoa rieng anh huong TAP CA
print(f"\n{'lop':<14}{'n_all':>7}{'prize_all':>11}{'n_sub':>7}{'prize_sub':>11}{'chenh':>9}")
for c in ("femoral_cart", "med_tib_cart"):
    all_v = np.array([v for (cs, cl), v in m0.items() if cl == c], float)
    all_v = all_v[np.isfinite(all_v)]
    sub_v = np.array([m0[x] for x in shared if x[1] == c], float)
    sub_v = sub_v[np.isfinite(sub_v)]
    if all_v.size and sub_v.size:
        print(f"{c:<14}{all_v.size:>7}{all_v.mean():>11.4f}{sub_v.size:>7}"
              f"{sub_v.mean():>11.4f}{all_v.mean() - sub_v.mean():>+9.4f}")

print("\nDoc ket qua:")
print("  max|A-B| ~0 (<1e-6) => hai pipeline tinh GIONG HET tren cung ca. Chenh lech giua")
print("     hai bang hoan toan do TAP CA (15 vs 103); bang 2 luong hoa dung phan do.")
print("  max|A-B| lon        => co khac biet THUC trong luat loc / NaN / contract be mat.")
print("     Luu y da biet: cell eps doi them pr_c.any(), Gate 1 thi khong.")

In [ ]:
# ---- MAU SO: cong M0 do tren ASSD TONG, nhung plan §3.7 do tren VUNG MONG -----
# §3.7 dieu 1: "cai thien outer-boundary error TRONG VUNG SUN MONG ~10% tuong doi tro len".
# Prize M0 = do giam ASSD TOAN BE MAT khi triet tieu loi vung mong => KHAC MAU SO.
# Cell nay doc thang M0_percase.jsonl (khong tinh lai gi) de quy doi giua hai mau so.
import json
import numpy as np
from bsc import metrics

M0_CKPT = f"{BSC_ROOT}/runs/M0_percase.jsonl"
THIN = ("absent", "<=0.5mm", "<=1.0mm")
GATE_PROCEED = 0.08   # nguong M0 hien hanh, tren ASSD TONG

per = {}
with open(M0_CKPT) as f:
    for line in f:
        r = json.loads(line)
        pb = r["mass"]["per_bin"]
        thin_n = sum(pb[b]["n"] for b in THIN)
        thin_mass = sum(pb[b]["error_mass_mm"] for b in THIN)
        tot_n = sum(pb[b]["n"] for b in pb)
        if thin_n == 0 or tot_n == 0:
            continue
        per.setdefault(r["cls"], []).append({
            "thin_err": thin_mass / thin_n,          # loi TB trong vung mong (mm)
            "thin_share": thin_mass / tot_n,         # tran ASSD tong neu xoa het loi vung mong
            "prize": r["prize"]["d_assd_prize_mm"],  # tran do duoc (counterfactual)
        })

def _ci(a):
    b = metrics.paired_bootstrap(np.zeros_like(a), a, n_boot=10000, seed=0)
    return b["ci_low"], b["ci_high"]

print("MAU SO 1 - VUNG MONG (dung theo plan §3.7 dieu 1)")
print(f"{'lop':<14}{'n':>4}{'loi TB vung mong':>19}{'muc tieu 10%':>15}")
tgt_overall = {}
for c, rows in per.items():
    te = np.array([r["thin_err"] for r in rows], float)
    ts = np.array([r["thin_share"] for r in rows], float)
    tgt_overall[c] = 0.10 * ts          # quy doi muc tieu 10% sang ASSD TONG
    lo, hi = _ci(te)
    print(f"{c:<14}{te.size:>4}{te.mean():>14.4f}mm{0.10*te.mean():>13.4f}mm   CI loi [{lo:.3f},{hi:.3f}]")

print("\nMAU SO 2 - ASSD TONG (cai cong M0 dang dung)")
print(f"{'lop':<14}{'tran do duoc':>14}{'muc tieu §3.7':>15}{'% tran can bat':>16}")
for c, rows in per.items():
    pz = np.array([r["prize"] for r in rows], float)
    pz = pz[np.isfinite(pz)]
    t = tgt_overall[c].mean()
    print(f"{c:<14}{pz.mean():>12.4f}mm{t:>13.4f}mm{t/pz.mean():>15.0%}")

print("\nCONG M0 HIEN HANH DOI GI, quy ve mau so vung mong?")
print(f"{'lop':<14}{'tran toi da':>13}{'nguong 0.08':>13}{'=> doi % vung mong':>21}  kha thi?")
for c, rows in per.items():
    ts = np.array([r["thin_share"] for r in rows], float).mean()   # tran ly thuyet
    need = GATE_PROCEED / ts
    feasible = "KHONG THE DAT" if need > 1.0 else f"can {need:.0%} - gan nhu hoan hao"
    print(f"{c:<14}{ts:>11.4f}mm{GATE_PROCEED:>12.2f}mm{need:>20.0%}  {feasible}")

print("\nDoc ket qua:")
print("  Cot cuoi > 100% => nguong 0.08 doi NHIEU HON toan bo loi ton tai o vung mong,")
print("     tuc KHONG PHUONG PHAP NAO dat duoc, du hoan hao. Do la nguong HONG,")
print("     ket luan nay doc lap voi moi ket qua do duoc.")
print("  So sanh 'muc tieu §3.7' voi 'tran do duoc' moi la phep thu dung cua plan.")

## Ghi kết quả M0 vào Drive (§7)

Dù quyết định là gì, **lưu lại** — kết quả âm tính M0 vẫn publishable (cùng ROI cascade).

In [ ]:
import numpy as np
now = None   # notebook: co the dat chuoi thoi gian thu cong neu muon
out = {c: {"gate1": headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])}
       for c in CART}
# ep numpy -> float cho json
def clean(o):
    if isinstance(o, dict): return {k:clean(v) for k,v in o.items()}
    if isinstance(o, (list,tuple)): return [clean(x) for x in o]
    if isinstance(o, (np.floating,np.integer)): return float(o)
    return o
path = f"{BSC_ROOT}/runs/M0_headroom_B0_ zibTs.json".replace(" ","")
json.dump(clean(out), open(path,"w"), indent=2)
print("Da ghi", path)
print("\nNEU PROCEED -> Phase 2 (geometry QC: M3/M2/M4 tren xuong that).")
print("NEU RESCOPE  -> viet lai muc tieu §3.7 quanh presence F1 + thickness MAE.")
print("NEU STOP     -> cong bo ket qua am tinh. Khong dot them chu ky.")